# RunGap Corridor Environment Visualization

This notebook demonstrates the RunGap corridor environment with:
1. Arena rendering (overview of the gap corridor)
2. Zero-action rollout with reward/position plots
3. Video from `close_profile-rodent` camera
4. **JAX-native egocentric vision** using the warp GPU ray-tracer (`JaxVisionRenderer`)
5. Concatenated video with vision overlay in upper-left corner

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
import jax
import jax.numpy as jp
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
from tqdm import tqdm

from vnl_playground.tasks.rodent import run_gap

## 1. Initialize Environment and Render Arena

In [ ]:
env = run_gap.RunGap()
mj_model = env.mj_model
fps = int(1.0 / env.dt)

print(f"Action size: {env.action_size}")
print(f"Obs size: {env.observation_size}")
print(f"Corridor end: {env._corridor_end_x:.2f}m")
print(f"Num platforms: {len(env._platform_positions)}")
print(f"FPS: {fps}")
print(f"Cameras: {[mj_model.camera(i).name for i in range(mj_model.ncam)]}")

In [ ]:
# Render a few static views of the arena
mj_data = mujoco.MjData(mj_model)
mujoco.mj_forward(mj_model, mj_data)

renderer = mujoco.Renderer(mj_model, height=480, width=640)

# Overview camera positions along the corridor
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.distance = 3.0
cam.elevation = -40

positions = [
    0.0,
    env._corridor_end_x / 3,
    2 * env._corridor_end_x / 3,
    env._corridor_end_x,
]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, x_pos in zip(axes, positions):
    cam.azimuth = 90
    cam.lookat[:] = [x_pos, 0.0, 0.0]
    renderer.update_scene(mj_data, camera=cam)
    ax.imshow(renderer.render())
    ax.set_title(f"x = {x_pos:.1f}m")
    ax.axis("off")
plt.suptitle("Arena Overview - Side Views Along Corridor", fontsize=14)
plt.tight_layout()
plt.show()

# Top-down view of full corridor
cam.distance = 8.0
cam.elevation = -89
cam.azimuth = 0
cam.lookat[:] = [env._corridor_end_x / 2, 0.0, 0.0]
renderer.update_scene(mj_data, camera=cam)
fig, ax = plt.subplots(1, 1, figsize=(16, 3))
ax.imshow(renderer.render())
ax.set_title("Top-Down View of Corridor")
ax.axis("off")
plt.tight_layout()
plt.show()

renderer.close()

## 2. Zero-Action Rollout

In [ ]:
rng = jax.random.PRNGKey(0)
state = jax.jit(env.reset)(rng)
step_fn = jax.jit(env.step)

n_steps = 100
rollout_states = []
qposes = [np.array(state.data.qpos)]
rewards = []
positions = []

torso = state.data.bind(env.mjx_model, env._spec.body("torso-rodent"))
positions.append(np.array(torso.xpos))

for i in tqdm(range(n_steps), desc="Zero-action rollout"):
    action = jp.zeros(env.action_size)
    state = step_fn(state, action)
    rollout_states.append(state)
    qposes.append(np.array(state.data.qpos))
    rewards.append(float(state.reward))
    torso = state.data.bind(env.mjx_model, env._spec.body("torso-rodent"))
    positions.append(np.array(torso.xpos))
    if state.done > 0.5:
        break

qposes = np.array(qposes)
rewards = np.array(rewards)
positions = np.array(positions)

print(f"Steps: {len(rewards)}")
print(f"Mean reward: {rewards.mean():.4f}")
print(f"Final x-pos: {positions[-1, 0]:.4f}m")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(rewards)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Reward")
axes[0].set_title("Reward over time")

axes[1].plot(positions[:, 0], label="X")
axes[1].plot(positions[:, 1], label="Y")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Position (m)")
axes[1].set_title("XY Position")
axes[1].legend()

axes[2].plot(positions[:, 2])
axes[2].axhline(y=-0.05, color="r", linestyle="--", label="Fall threshold")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Z (m)")
axes[2].set_title("Height over time")
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Video from close_profile-rodent Camera

In [ ]:
mj_data = mujoco.MjData(mj_model)
renderer = mujoco.Renderer(mj_model, height=480, width=640)

render_every = 2
frames_profile = []
for qpos in tqdm(qposes[::render_every], desc="Rendering close_profile"):
    mj_data.qpos = qpos
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data, camera="close_profile-rodent")
    frames_profile.append(renderer.render().copy())

renderer.close()
print(f"Rendered {len(frames_profile)} frames")
media.show_video(frames_profile, fps=fps // render_every)

## 4. JAX-Native Egocentric Vision (Warp GPU Ray-Tracer)

This uses the new `JaxVisionRenderer` from `vision_jax.py`, which wraps the
`create_mjx_render_fn` factory from the MJX warp backend. The renderer:

- Is **JAX-traceable** — works inside `jax.jit` and `jax.lax.scan`
- Captures static warp state (Model, Data, RenderContext) in a closure
- Only dynamic kinematic arrays flow through JAX
- Produces float32 images in [0, 1] (grayscale or RGB)

This is the same renderer used during training via `VisionRenderWrapper`.

In [ ]:
from vnl_playground.tasks.rodent.vision_jax import JaxVisionRenderer
from mujoco.mjx.warp import render_jax

# Create renderer for batched MJX rollout
# nworld=1 since our rollout is single-world
NWORLD = 1
WIDTH, HEIGHT = 64, 64

vision_renderer = JaxVisionRenderer(
    mj_model=mj_model,
    nworld=NWORLD,
    width=WIDTH,
    height=HEIGHT,
    grayscale=False,  # RGB for visualization
    camera_name="egocentric-rodent",
)

print(f"JaxVisionRenderer initialized: {NWORLD} world(s), {WIDTH}x{HEIGHT}")
print(f"Vision shape: {vision_renderer.vision_shape}")
print(f"Render info: {vision_renderer.info}")

In [ ]:
# Render vision from the Section 2 rollout states using JaxVisionRenderer.
#
# The rollout states already contain warp-backed mjx.Data with all kinematic
# arrays (geom_xpos, cam_xpos, etc.) populated from the physics step.
# We just need to add a batch dimension (nworld=1) for the renderer.

n_vision_steps = min(500, len(rollout_states))


def add_batch_dim(data):
    """Add a leading nworld=1 dimension to all arrays in mjx.Data."""
    return jax.tree.map(lambda x: x[None, ...], data)


@jax.jit
def render_from_data(data):
    """Render from mjx.Data (already has kinematic arrays from physics step)."""
    batched_data = add_batch_dim(data)
    return vision_renderer.render(batched_data)


print(
    f"Rendering {n_vision_steps // render_every} vision frames via JaxVisionRenderer..."
)

# Render vision for each saved state from the Section 2 rollout
vision_images = []
for state in tqdm(
    rollout_states[:n_vision_steps:render_every], desc="JAX vision render"
):
    images = render_from_data(state.data)  # (1, H, W, 3)
    vision_images.append(np.array(images[0]))  # take world 0

print(f"Collected {len(vision_images)} vision frames")
print(f"Frame shape: {vision_images[0].shape}, dtype: {vision_images[0].dtype}")
print(f"Value range: [{vision_images[0].min():.3f}, {vision_images[0].max():.3f}]")

In [ ]:
# Show sample vision frames
sample_indices = np.linspace(0, len(vision_images) - 1, 10, dtype=int)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(vision_images[idx])
    ax.set_title(f"Step {idx * render_every}")
    ax.axis("off")
plt.suptitle(
    f"Egocentric Camera ({WIDTH}x{HEIGHT}) — JAX-native Warp Renderer", fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
# Compare: MuJoCo CPU renderer vs JAX-native Warp renderer (egocentric camera)
mj_data_cmp = mujoco.MjData(mj_model)
renderer_cmp = mujoco.Renderer(mj_model, height=HEIGHT, width=WIDTH)

compare_indices = [0, len(rollout_states) // 4, len(rollout_states) // 2]
fig, axes = plt.subplots(2, len(compare_indices), figsize=(4 * len(compare_indices), 8))

for col, idx in enumerate(compare_indices):
    state = rollout_states[idx]
    mj_data_cmp.qpos = np.array(state.data.qpos)
    mujoco.mj_forward(mj_model, mj_data_cmp)

    # Row 0: MuJoCo CPU renderer (egocentric camera)
    renderer_cmp.update_scene(mj_data_cmp, camera="egocentric-rodent")
    axes[0, col].imshow(renderer_cmp.render().copy())
    axes[0, col].set_title(f"Step {idx}")
    axes[0, col].axis("off")

    # Row 1: JAX-native Warp renderer (same state)
    warp_img = render_from_data(state.data)  # (1, H, W, 3)
    axes[1, col].imshow(np.array(warp_img[0]))
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("MuJoCo CPU\nRenderer", fontsize=11)
axes[1, 0].set_ylabel("Warp GPU\n(JAX-native)", fontsize=11)
plt.suptitle("Egocentric Camera Comparison: MuJoCo CPU vs Warp GPU", fontsize=14)
plt.tight_layout()
plt.show()

renderer_cmp.close()

In [ ]:
# Show vision as video
vision_uint8 = [np.clip(f * 255, 0, 255).astype(np.uint8) for f in vision_images]
media.show_video(
    vision_uint8,
    fps=fps // render_every,
    title=f"Egocentric Vision ({WIDTH}x{HEIGHT}, JAX-native Warp Renderer)",
)

## 5. Concatenated Video: Scene + Warp Vision Overlay

The JAX-native egocentric vision output is overlaid on the upper-left corner of the
main scene render.

In [ ]:
def overlay_vision_on_frame(scene_frame, vision_frame, scale=3, padding=8, border=2):
    """Overlay a small vision frame on the upper-left corner of a scene frame.

    Args:
        scene_frame: (H, W, 3) uint8 main camera frame.
        vision_frame: (h, w, 3) float32 [0,1] or uint8 egocentric vision frame.
        scale: Upscale factor for the vision inset.
        padding: Pixel padding from the top-left corner.
        border: Border width around the vision inset.

    Returns:
        (H, W, 3) uint8 frame with vision overlay.
    """
    frame = scene_frame.copy()

    # Convert float32 [0,1] to uint8 if needed
    if vision_frame.dtype != np.uint8:
        vision_frame = np.clip(vision_frame * 255, 0, 255).astype(np.uint8)

    vh, vw = vision_frame.shape[:2]
    sh, sw = vh * scale, vw * scale

    # Upscale vision frame using nearest-neighbor
    vision_up = np.kron(vision_frame, np.ones((scale, scale, 1))).astype(np.uint8)

    # Draw border (dark background)
    y0 = padding
    x0 = padding
    frame[
        y0 - border : y0 + sh + border,
        x0 - border : x0 + sw + border,
    ] = 32  # dark gray border

    # Paste vision
    frame[y0 : y0 + sh, x0 : x0 + sw] = vision_up

    return frame

In [ ]:
# Build concatenated frames
n_concat = min(len(frames_profile), len(vision_images))
concat_frames = []
for i in range(n_concat):
    concat_frames.append(
        overlay_vision_on_frame(frames_profile[i], vision_images[i], scale=3)
    )

print(f"Concatenated {n_concat} frames")
media.show_video(
    concat_frames,
    fps=fps // render_every,
    title="Scene + JAX-native Egocentric Vision Overlay",
)

In [ ]:
# Save all videos to disk
output_dir = os.path.join(os.getcwd(), "notebooks")

media.write_video(
    os.path.join(output_dir, "run_gap_close_profile.mp4"),
    frames_profile,
    fps=fps // render_every,
)
media.write_video(
    os.path.join(output_dir, "run_gap_egocentric_vision.mp4"),
    vision_uint8,
    fps=fps // render_every,
)
media.write_video(
    os.path.join(output_dir, "run_gap_scene_with_vision.mp4"),
    concat_frames,
    fps=fps // render_every,
)

print(f"Saved videos to {output_dir}/")